# DLP NPPE-2 - Uyghur Automatic Speech Recognition (ASR)

## 1. Problem Statement
We are given **23.95 hours** of Uyghur speech (9,468 mono `.wav` clips @ 16kHz) split into:
- `train.csv` — 7,574 samples with `ID`, `filepath`, `transcription`
- `test.csv` — 1,894 samples with `ID`, `filepath` (no transcription — to predict)

**Goal:** train a model that listens to an audio clip and outputs the correct Uyghur transcription (written in a Latin transliteration). No restrictions on model, compute, or external data.

## 2. Evaluation Metric :- Character Error Rate (CER)
$$CER = \frac{S + D + I}{N}$$
`S`=substitutions, `D`=deletions, `I`=insertions (character-level Levenshtein distance), `N`=reference length. Lower is better; 0 = perfect.

## 3. Approach
| Choice | Reason |
|---|---|
| **Model** | `facebook/mms-300m` — native Uyghur (`uig`) coverage, better than Whisper for this low-resource language |
| **Task head** | CTC — lighter/faster than seq2seq, natural fit for a character-level target vocabulary |
| **Tokenizer** | Custom character-level vocab built directly from the training transcriptions |
| **Precision** | Mixed precision (`torch.cuda.amp`) + gradient checkpointing — this is the proven, working configuration that trains at ~23-25 min/epoch |
| **Training budget (this version)** | Target **30 epochs**, bounded by an **8-hour training time budget**, hard-capped at **8.5 hours total** (training + inference + submission write) so a valid `submission.csv` is always produced |

This notebook is the **exact working training pipeline**, unchanged — the only additions are `NUM_EPOCHS = 30` and the time-budget stop logic wrapped around the same training loop.

## 4. Notebook Structure
1. Configuration, packages, imports, seed, GPU + **time-budget setup**
2. Data loading & exploratory analysis
3. Custom tokenizer & processor
4. Model loading
5. Audio loader, Dataset & Collator
6. CER / decoding utilities
7. Dry run sanity check
8. Optimizer, scheduler, validation function
9. **Training loop (time-budget aware)**
10. Inference on test set
11. Submission generation & verification


In [1]:
# ============================================================
# UYGHUR ASR — NPPE-2 | facebook/mms-300m | CTC Fine-tune
# MMS has native Uyghur — better than Whisper for this task
# ============================================================

# SECTION 1: CONFIGURATION
import os, warnings, subprocess, sys
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ROOT       = "/kaggle/input/competitions/dlp-26-t-2-nppe-2"
TRAIN_CSV  = f"{ROOT}/train.csv"
TEST_CSV   = f"{ROOT}/test.csv"
OUTPUT_DIR = "/kaggle/working"
CKPT_DIR   = f"{OUTPUT_DIR}/best_mms"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

SEED              = 42
# MMS model — native Uyghur support
MODEL_NAME        = "facebook/mms-300m"
TARGET_LANG       = "uig"          # Uyghur ISO 639-3 code for MMS
NUM_EPOCHS        = 30             # <-- CHANGED from 10 to 30 (target upper bound)
BATCH_SIZE        = 4          # CTC much lighter than seq2seq
GRAD_ACCUM_STEPS  = 8              # eff batch = 32
LEARNING_RATE     = 1e-4
WARMUP_STEPS      = 150
MAX_AUDIO_SECONDS = 30.0
VALIDATION_RATIO  = 0.10
SAMPLE_RATE       = 16_000
NUM_WORKERS       = 4
PATIENCE          = 3
VAL_EVERY_N       = 2
PRINT_EVERY_PCT   = [0.25, 0.5, 0.75, 1.0]

# --- TIME BUDGET ADDITION ---
TRAIN_TIME_BUDGET_HOURS = 7.8     # stop TRAINING once this much wall-clock time has passed
HARD_STOP_HOURS         = 8.5     # absolute ceiling incl. inference + submission write

# SECTION 2: PACKAGES
def pip_q(*pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p],
                       check=True)

try: import jiwer
except ImportError: print("[INSTALL] jiwer"); pip_q("jiwer")
try: import soundfile
except ImportError: print("[INSTALL] soundfile"); pip_q("soundfile")

# SECTION 3: IMPORTS
import random, time, gc
import numpy as np
import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import List, Optional, Dict
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from jiwer import cer as jiwer_cer
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    get_cosine_schedule_with_warmup,
)

# SECTION 4: SEED
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

# SECTION 5: GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def print_gpu(label=""):
    if not torch.cuda.is_available(): return
    alloc = torch.cuda.memory_allocated(0)/1e9
    resrv = torch.cuda.memory_reserved(0)/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"  [GPU {label}] alloc={alloc:.2f}GB  reserved={resrv:.2f}GB  "
          f"free≈{total-resrv:.2f}GB  total={total:.2f}GB")

TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory/1e9 \
             if torch.cuda.is_available() else 0
GPU_NAME   = torch.cuda.get_device_properties(0).name \
             if torch.cuda.is_available() else "CPU"

print("="*60)
print(f"  DEVICE : {DEVICE}  |  GPU: {GPU_NAME}")
print(f"  VRAM   : {TOTAL_VRAM:.2f} GB")
print(f"  MODEL  : {MODEL_NAME} (native Uyghur!)")
print(f"  BATCH  : {BATCH_SIZE} (eff={BATCH_SIZE*GRAD_ACCUM_STEPS})")
print("="*60)
print_gpu("startup")

# --- TIME BUDGET ADDITION ---
# SECTION 5b: TIME BUDGET SETUP
NOTEBOOK_START_TIME   = time.time()
TRAIN_TIME_BUDGET_SEC = TRAIN_TIME_BUDGET_HOURS * 3600
HARD_STOP_SEC         = HARD_STOP_HOURS * 3600

def elapsed_hours():
    return (time.time() - NOTEBOOK_START_TIME) / 3600

def time_budget_exceeded():
    return (time.time() - NOTEBOOK_START_TIME) >= TRAIN_TIME_BUDGET_SEC

print(f"  TIME BUDGET: train up to {TRAIN_TIME_BUDGET_HOURS}h, "
      f"hard stop at {HARD_STOP_HOURS}h (incl. inference)")


[INSTALL] jiwer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 90.2 MB/s eta 0:00:00
  DEVICE : cuda  |  GPU: Tesla T4
  VRAM   : 15.64 GB
  MODEL  : facebook/mms-300m (native Uyghur!)
  BATCH  : 4 (eff=32)
  [GPU startup] alloc=0.00GB  reserved=0.00GB  free≈15.64GB  total=15.64GB
  TIME BUDGET: train up to 7.8h, hard stop at 8.5h (incl. inference)


## Data Loading & Exploratory Analysis

Load train/test metadata, inspect transcription length and character vocabulary, then split off a validation set.

In [2]:
# SECTION 6: CSVs
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
train_df["audio_path"] = train_df["filepath"].apply(
    lambda x: os.path.join(ROOT, x))
test_df["audio_path"]  = test_df["filepath"].apply(
    lambda x: os.path.join(ROOT, x))
train_df = train_df.dropna(subset=["transcription"]).reset_index(drop=True)
print(f"\n  Train: {len(train_df)} | Test: {len(test_df)}")
print(f"  Sample: {train_df['transcription'].iloc[0]}")

# SECTION 7: INSPECT VOCAB
tlen  = train_df["transcription"].str.len()
vocab = set("".join(train_df["transcription"].tolist()))
print(f"\n  Transcription: mean={tlen.mean():.1f} max={tlen.max()} "
      f"95th={tlen.quantile(0.95):.1f}")
print(f"  Vocab ({len(vocab)} chars): {''.join(sorted(vocab))}")

# SECTION 8: SPLIT
train_split, val_split = train_test_split(
    train_df, test_size=VALIDATION_RATIO, random_state=SEED, shuffle=True
)
train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)
print(f"\n  Train: {len(train_split)} | Val: {len(val_split)}")



  Train: 7574 | Test: 1894
  Sample: vuyGur HAlqiniN fevudal bAglArniN wA pomexciklar sinipiniN hOkUmranliqidiki jAbir japaliq turmuxini vAks vAttUridiGan qoxaqlar bar

  Transcription: mean=113.4 max=747 95th=252.0
  Vocab (34 chars):  AGHJNOUabcdefghijklmnopqrstuvwxyz

  Train: 6816 | Val: 758


**Insight:** the vocabulary is a small Latin character set, so a character-level CTC head is a natural fit, no subword tokenizer needed. With ~7.5K training clips this is a genuinely low-resource setting, which is exactly the regime `mms-300m`'s multilingual pre-training helps with.

## Custom Tokenizer & Processor

Build a character-level vocabulary directly from the **training-only** transcriptions (never test), wrap it in a `Wav2Vec2CTCTokenizer` + `Wav2Vec2FeatureExtractor` pair.

In [3]:
# ============================================================
# SECTION 9: CUSTOM TOKENIZER
# Build from training transcriptions — preserve exact case
# MMS-300m uses Wav2Vec2CTCTokenizer with custom vocab
# ============================================================
print("\n[TOKENIZER] Building custom vocab from train transcriptions...")

# Collect all unique characters from train only (never test)
all_chars = set()
for text in train_split["transcription"].tolist():
    all_chars.update(list(str(text)))

# Sort for reproducibility
vocab_list = sorted(list(all_chars))

# Special tokens for CTC
# pad = blank token (CTC requires this)
special_tokens = ["<pad>", "<unk>", "|"]  # | = word boundary/space
vocab_dict = {v: i for i, v in enumerate(special_tokens)}

# Add all characters
for ch in vocab_list:
    if ch not in vocab_dict:
        vocab_dict[ch] = len(vocab_dict)

print(f"  Vocab size: {len(vocab_dict)}")
print(f"  Chars: {''.join(sorted(all_chars))}")

# Save vocab json for tokenizer
import json
VOCAB_PATH = f"{OUTPUT_DIR}/vocab.json"
with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)
print(f"  Vocab saved → {VOCAB_PATH}")

# Build tokenizer
tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_PATH,
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|",
)

# Verify
sample_text = train_split["transcription"].iloc[0]
encoded     = tokenizer(sample_text).input_ids
decoded     = tokenizer.decode(encoded)
print(f"\n  Sample encode/decode check:")
print(f"  IN : {sample_text[:60]}")
print(f"  OUT: {decoded[:60]}")
print(f"  Match: {sample_text == decoded}")

# SECTION 10: FEATURE EXTRACTOR
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=SAMPLE_RATE,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)
print("\n  ✓ Processor ready")



[TOKENIZER] Building custom vocab from train transcriptions...
  Vocab size: 37
  Chars:  AGHJNOUabcdefghijklmnopqrstuvwxyz
  Vocab saved → /kaggle/working/vocab.json

  Sample encode/decode check:
  IN : bu yolda vaylanma bAk kOp hadisA vasan yUz beridu xopurlar v
  OUT: bu yolda vaylanma bAk kOp hadisA vasan yUz beridu xopurlar v
  Match: True

  ✓ Processor ready


In [4]:
# SECTION 11: MODEL
print(f"\n[MODEL] Loading {MODEL_NAME}...")
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ignore_mismatched_sizes=True,  # CTC head will be replaced
    ctc_loss_reduction="mean",
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(vocab_dict),
)

# Freeze feature encoder — only fine-tune transformer layers
# This is the standard approach for MMS/Wav2Vec2 fine-tuning
model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)
print(f"  Total params    : {total_params/1e6:.1f}M")
print(f"  Trainable       : {trainable_params/1e6:.1f}M "
      f"({100*trainable_params/total_params:.1f}%)")

model = model.to(DEVICE)
print_gpu("after model load")
print("  ✓ Model ready")



[MODEL] Loading facebook/mms-300m...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total params    : 315.5M
  Trainable       : 311.3M (98.7%)
  [GPU after model load] alloc=1.26GB  reserved=1.28GB  free≈14.35GB  total=15.64GB
  ✓ Model ready


In [5]:
# SECTION 12: AUDIO LOADER
def load_audio(path: str) -> Optional[torch.Tensor]:
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        return wav.squeeze(0)
    except Exception as ex:
        print(f"  [WARN] {os.path.basename(path)}: {ex}")
        return None

# SECTION 13: DATASET
class UyghurDataset(Dataset):
    def __init__(self, df, processor, is_train=True):
        self.df        = df
        self.processor = processor
        self.is_train  = is_train
        self.max_len   = int(MAX_AUDIO_SECONDS * SAMPLE_RATE)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = load_audio(row["audio_path"])
        if wav is None or len(wav) < 400:
            wav = torch.zeros(SAMPLE_RATE, dtype=torch.float32)
        if len(wav) > self.max_len:
            wav = wav[:self.max_len]

        # Feature extraction — returns float32
        inputs = self.processor.feature_extractor(
            wav.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
        )
        item = {
            "input_values":   inputs.input_values.squeeze(0),
            "attention_mask": inputs.attention_mask.squeeze(0),
        }

        if self.is_train:
            # Direct tokenizer call — as_target_processor deprecated
            item["labels"] = self.processor.tokenizer(
                str(row["transcription"])
            ).input_ids
        return item

# SECTION 14: COLLATOR (CTC-style)
@dataclass
class CTCCollator:
    processor: Wav2Vec2Processor
    padding: bool = True

    def __call__(self, features: List[Dict]) -> Dict:
        # Separate inputs and labels
        input_features = [
            {"input_values":   f["input_values"],
             "attention_mask": f["attention_mask"]}
            for f in features
        ]
        batch = self.processor.pad(
            input_features,
            padding=True,
            return_tensors="pt",
        )

        if "labels" in features[0]:
            label_features = [{"input_ids": f["labels"]} for f in features]
            # FIX: as_target_processor() deprecated — use tokenizer directly
            labels_batch = self.processor.tokenizer.pad(
                label_features,
                padding=True,
                return_tensors="pt",
            )
            # Replace padding with -100 so CTC loss ignores it
            labels = labels_batch["input_ids"].masked_fill(
                labels_batch.attention_mask.ne(1), -100
            )
            batch["labels"] = labels

        return batch

collator = CTCCollator(processor)
pin      = (DEVICE.type == "cuda")

train_loader = DataLoader(
    UyghurDataset(train_split, processor, True),
    batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collator, num_workers=NUM_WORKERS,
    pin_memory=pin, prefetch_factor=2, persistent_workers=True)

val_loader = DataLoader(
    UyghurDataset(val_split, processor, True),
    batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collator, num_workers=NUM_WORKERS,
    pin_memory=pin, prefetch_factor=2, persistent_workers=True)

test_loader = DataLoader(
    UyghurDataset(test_df, processor, False),
    batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collator, num_workers=NUM_WORKERS,
    pin_memory=pin, prefetch_factor=2, persistent_workers=True)

print(f"\n  Batches → Train:{len(train_loader)} "
      f"Val:{len(val_loader)} Test:{len(test_loader)}")



  Batches → Train:1704 Val:190 Test:474


## CER & Decoding Utilities

Character Error Rate computation (via `jiwer`) and greedy CTC decoding, with the `|` word-delimiter token converted back to spaces.

In [6]:
# SECTION 15: CER
def compute_cer(refs: List[str], hyps: List[str]) -> float:
    r = [x if x.strip() else " " for x in refs]
    h = [x if x.strip() else " " for x in hyps]
    return jiwer_cer(r, h)

def decode_predictions(logits):
    '''Greedy CTC decode — remove word delimiter | from output.'''
    pred_ids = torch.argmax(logits, dim=-1)
    pred_str = processor.batch_decode(pred_ids)
    # MMS uses | as word delimiter — replace with space
    pred_str = [p.replace("|", " ").strip() for p in pred_str]
    return pred_str


## Dry Run

Sanity-check a handful of train/val steps (forward, backward, CTC decode) before committing the full 8-hour time budget to training.

In [7]:
# ============================================================
# SECTION 16: DRY RUN
# ============================================================
print("\n"+"="*60)
print("  DRY RUN — 5 train + 3 val")
print("="*60)

model.train()
opt_dry    = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE)
scaler_dry = torch.cuda.amp.GradScaler()
opt_dry.zero_grad()

dry_iter = iter(train_loader)
for i in range(5):
    batch = next(dry_iter)
    input_values   = batch["input_values"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels         = batch["labels"].to(DEVICE)

    with torch.cuda.amp.autocast():
        out  = model(input_values=input_values,
                     attention_mask=attention_mask,
                     labels=labels)
        loss = out.loss / GRAD_ACCUM_STEPS

    scaler_dry.scale(loss).backward()
    if i == 4:
        scaler_dry.unscale_(opt_dry)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        scaler_dry.step(opt_dry); scaler_dry.update()
        opt_dry.zero_grad()
    print(f"  [DryTrain {i+1}/5] loss={loss.item()*GRAD_ACCUM_STEPS:.4f}")

del opt_dry, scaler_dry
torch.cuda.empty_cache()

model.eval()
print("\n  [DryVal] Testing CTC decode...")
dv_iter = iter(val_loader)
with torch.no_grad():
    for i in range(3):
        batch          = next(dv_iter)
        input_values   = batch["input_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast():
            out = model(input_values=input_values,
                        attention_mask=attention_mask,
                        labels=labels)

        preds   = decode_predictions(out.logits)
        ref_ids = labels.masked_fill(
            labels == -100, tokenizer.pad_token_id)
        refs    = tokenizer.batch_decode(ref_ids, skip_special_tokens=True)

        print(f"  [DryVal {i+1}/3] loss={out.loss.item():.4f}")
        print(f"    REF : {refs[0][:80]}")
        print(f"    PRED: {preds[0][:80]}")

print_gpu("after dry run")
print("\n  ✓ DRY RUN PASSED\n")
torch.cuda.empty_cache(); gc.collect()



  DRY RUN — 5 train + 3 val
  [DryTrain 1/5] loss=14.1877
  [DryTrain 2/5] loss=10.7500
  [DryTrain 3/5] loss=12.4463
  [DryTrain 4/5] loss=12.5992
  [DryTrain 5/5] loss=12.4011

  [DryVal] Testing CTC decode...
  [DryVal 1/3] loss=12.1224
    REF : xah mAHsut waNniN beyjiNdiki qumul sariyi niN vorni beyjiN xUnwomin vicidiki tAy
    PRED: 
  [DryVal 2/3] loss=9.1569
    REF : xu kUni bir miN toquz yUz yAtmix yAtinci yili Harwardni pUtUrgAn mikro sofitniN 
    PRED: v
  [DryVal 3/3] loss=13.1078
    REF : vuniN pul HAjlAp bu nArsilArni velixqa kOzi qiymaydu
    PRED: 
  [GPU after dry run] alloc=1.29GB  reserved=2.76GB  free≈12.88GB  total=15.64GB

  ✓ DRY RUN PASSED



364

## Optimizer, Scheduler & Validation Function

AdamW + cosine schedule with warmup, and a full validation-loop helper that returns loss, CER, and 3 example predictions.

In [8]:
# SECTION 17: OPTIMIZER + SCHEDULER
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=1e-2
)
total_opt_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_opt_steps,
)
scaler = torch.cuda.amp.GradScaler()
print(f"  Opt steps: {total_opt_steps} | Warmup: {WARMUP_STEPS}")

# SECTION 18: VALIDATION FUNCTION
def run_validation(mdl, ldr, n_examples=3):
    mdl.eval()
    total_loss=0.0; all_refs=[]; all_hyps=[]; ex_r=[]; ex_h=[]

    with torch.no_grad():
        for batch in tqdm(ldr, desc="  Val", leave=False):
            input_values   = batch["input_values"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            with torch.cuda.amp.autocast():
                out = mdl(input_values=input_values,
                          attention_mask=attention_mask,
                          labels=labels)
            total_loss += out.loss.item()

            preds   = decode_predictions(out.logits)
            ref_ids = labels.masked_fill(
                labels == -100, tokenizer.pad_token_id)
            refs    = tokenizer.batch_decode(
                ref_ids, skip_special_tokens=True)

            all_refs.extend(refs); all_hyps.extend(preds)
            if len(ex_r) < n_examples:
                ex_r.extend(refs); ex_h.extend(preds)

    print(f"\n  ── Examples ──")
    for i,(r,h) in enumerate(zip(ex_r[:n_examples], ex_h[:n_examples])):
        print(f"  [{i+1}] REF : {r}")
        print(f"       PRED: {h}")

    return total_loss/len(ldr), compute_cer(all_refs, all_hyps)


  Opt steps: 6390 | Warmup: 150


## Training Loop (Time-Budget Aware)

Same training loop as the original, working version — with the **time-budget stop logic** layered on top (marked `# --- TIME BUDGET ADDITION ---` in the code):

- Checked **before every epoch starts**, and **every 100 steps inside an epoch**, so one long epoch can't blow past the 8-hour budget.
- If the budget is hit or patience-based early stopping triggers, the loop breaks and always proceeds to inference afterward — so a checkpoint (and eventually `submission.csv`) is guaranteed.

In [9]:
# SECTION 19: TRAINING LOOP
TOTAL_STEPS = len(train_loader)
print_steps = sorted(set(
    [max(1, int(TOTAL_STEPS*p)) for p in PRINT_EVERY_PCT]))

print("\n"+"="*60)
print(f"  {MODEL_NAME} | CTC | {NUM_EPOCHS} epochs | BATCH={BATCH_SIZE}")
print(f"  eff_batch={BATCH_SIZE*GRAD_ACCUM_STEPS} | "
      f"Val every {VAL_EVERY_N} epochs")
print("="*60+"\n")

best_cer=float("inf"); patience_count=0; val_epochs=[]
print_gpu("before training")

# --- TIME BUDGET ADDITION ---
TIME_CHECK_EVERY_N_STEPS = 100
stop_training = False

for epoch in range(1, NUM_EPOCHS+1):
    # --- TIME BUDGET ADDITION ---
    if time_budget_exceeded():
        print(f"\n  [TIME BUDGET] {TRAIN_TIME_BUDGET_HOURS}h reached before "
              f"epoch {epoch} started. Stopping training.")
        break

    model.train()
    t_start=time.time(); train_loss=0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=TOTAL_STEPS,
                desc=f"Ep {epoch}/{NUM_EPOCHS}")

    actual_steps = TOTAL_STEPS  # --- TIME BUDGET ADDITION --- (updated if we break early)

    for step, batch in pbar:
        input_values   = batch["input_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast():
            out  = model(input_values=input_values,
                         attention_mask=attention_mask,
                         labels=labels)
            loss = out.loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        train_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step+1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()

        pbar.set_postfix({"loss": f"{train_loss/(step+1):.4f}"})

        if (step+1) in print_steps:
            pct     = int(100*(step+1)/TOTAL_STEPS)
            elapsed = (time.time()-t_start)/60
            print(f"\n  [Ep{epoch} {pct:3d}%] "
                  f"loss={train_loss/(step+1):.4f}  "
                  f"elapsed={elapsed:.1f}m")

        # --- TIME BUDGET ADDITION ---
        if (step+1) % TIME_CHECK_EVERY_N_STEPS == 0 and time_budget_exceeded():
            print(f"\n  [TIME BUDGET] {TRAIN_TIME_BUDGET_HOURS}h reached "
                  f"mid-epoch {epoch}. Stopping training now.")
            actual_steps = step + 1
            stop_training = True
            break

    # Leftover steps
    if TOTAL_STEPS % GRAD_ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step(); optimizer.zero_grad()

    avg_train = train_loss/actual_steps  # --- TIME BUDGET ADDITION --- (was /TOTAL_STEPS)
    train_t   = (time.time()-t_start)/60
    print(f"\n  Ep{epoch} TRAIN | loss={avg_train:.4f} | {train_t:.1f}m")
    print_gpu(f"ep{epoch}_train")

    # --- TIME BUDGET ADDITION --- (added "or stop_training" so we always get a final CER read)
    do_val = (epoch % VAL_EVERY_N == 0) or (epoch == NUM_EPOCHS) or stop_training
    if do_val:
        print(f"\n  Ep{epoch} — Validating...")
        val_loss, val_cer = run_validation(model, val_loader)
        total_t = (time.time()-t_start)/60
        val_epochs.append(epoch)

        print(f"\n  ── Ep{epoch:02d} RESULT ──")
        print(f"    train_loss = {avg_train:.4f}")
        print(f"    val_loss   = {val_loss:.4f}")
        print(f"    val_CER    = {val_cer:.4f}  ← KEY")
        print(f"    time       = {total_t:.1f}m")
        print_gpu(f"ep{epoch}_val")

        if val_cer < best_cer:
            best_cer=val_cer; patience_count=0
            model.save_pretrained(CKPT_DIR)
            processor.save_pretrained(CKPT_DIR)
            tokenizer.save_pretrained(CKPT_DIR)
            print(f"  ✓ NEW BEST CER={best_cer:.4f} → saved")
        else:
            patience_count+=1
            print(f"  No improvement. Patience {patience_count}/{PATIENCE}")
            if patience_count >= PATIENCE:
                print("  ⚡ Early stopping.")
                stop_training = True  # --- TIME BUDGET ADDITION --- (was: break)
    else:
        nxt = epoch + (VAL_EVERY_N - epoch % VAL_EVERY_N)
        print(f"  (Val skipped → next ep {nxt})")

    torch.cuda.empty_cache(); gc.collect()

    # --- TIME BUDGET ADDITION ---
    if stop_training:
        break

print(f"\n  ✓ Training done. Best CER={best_cer:.4f}")
print(f"  Validated at epochs: {val_epochs}")
print(f"  Total elapsed: {elapsed_hours():.2f}h")  # --- TIME BUDGET ADDITION ---

# --- TIME BUDGET ADDITION --- (safety net: guarantee a checkpoint exists for inference)
if not os.path.exists(os.path.join(CKPT_DIR, "config.json")):
    print("  No checkpoint saved yet — saving current model state as fallback so inference can proceed.")
    model.save_pretrained(CKPT_DIR)
    processor.save_pretrained(CKPT_DIR)
    tokenizer.save_pretrained(CKPT_DIR)



  facebook/mms-300m | CTC | 30 epochs | BATCH=4
  eff_batch=32 | Val every 2 epochs

  [GPU before training] alloc=1.29GB  reserved=1.35GB  free≈14.29GB  total=15.64GB


Ep 1/30:   0%|          | 0/1704 [00:00<?, ?it/s]


  [Ep1  25%] loss=7.3676  elapsed=6.3m

  [Ep1  50%] loss=5.4719  elapsed=12.8m

  [Ep1  75%] loss=4.7207  elapsed=19.5m

  [Ep1 100%] loss=4.3367  elapsed=25.9m

  Ep1 TRAIN | loss=4.3367 | 25.9m
  [GPU ep1_train] alloc=3.78GB  reserved=8.62GB  free≈7.02GB  total=15.64GB
  (Val skipped → next ep 2)


Ep 2/30:   0%|          | 0/1704 [00:00<?, ?it/s]


  [Ep2  25%] loss=3.1648  elapsed=6.3m

  [Ep2  50%] loss=3.1626  elapsed=12.9m

  [Ep2  75%] loss=3.1279  elapsed=19.4m

  [Ep2 100%] loss=2.8269  elapsed=26.0m

  Ep2 TRAIN | loss=2.8269 | 26.0m
  [GPU ep2_train] alloc=3.78GB  reserved=8.51GB  free≈7.12GB  total=15.64GB

  Ep2 — Validating...


...


  [Ep18  25%] loss=0.1586  elapsed=6.8m

  [Ep18  50%] loss=0.1544  elapsed=13.3m

  [TIME BUDGET] 7.8h reached mid-epoch 18. Stopping training now.

  Ep18 TRAIN | loss=0.1545 | 15.3m
  [GPU ep18_train] alloc=3.78GB  reserved=8.42GB  free≈7.21GB  total=15.64GB

  Ep18 — Validating...


  Val:   0%|          | 0/190 [00:00<?, ?it/s]


  ── Examples ──
  [1] REF : xah mAHsut waNniN beyjiNdiki qumul sariyi niN vorni beyjiN xUnwomin vicidiki tAypiNHu xiHuma kocisida vidi
       PRED: xah mAHsutwaNniN beyjiNdiki qumul sariyiniN vorni beyjiN xenwumin vicidiki tAypiNHu xiHuma kocisidar vidi
  [2] REF : xUmiNrUygA voHxax xen xAnmu kulub vAzaliq kartisini velip vArz qildi
       PRED: xu miN rUygA voHxax xinxAnmu kulub vAzaliq kartisini velip vArz qildi
  [3] REF : bowam gArcA yaxtin vexip qalGan bolsimu lekin yAnila nahayiti saGlam
       PRED: bowam gArcA yaHxti vexip qalGan bolsimu lekin yAnila nahayiti saGlam

  ── Ep18 RESULT ──
    train_loss = 0.1545
    val_loss   = 0.1640
    val_CER    = 0.0536  ← KEY
    time       = 16.4m
  [GPU ep18_val] alloc=3.78GB  reserved=8.42GB  free≈7.21GB  total=15.64GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ NEW BEST CER=0.0536 → saved

  ✓ Training done. Best CER=0.0536
  Validated at epochs: [2, 4, 6, 8, 10, 12, 14, 16, 18]
  Total elapsed: 7.84h


**Insight:** if training stopped because of the **time budget** rather than convergence or patience, `best_cer` still reflects the best checkpoint seen so far, and the fallback save above guarantees a checkpoint exists either way, the notebook proceeds to inference unconditionally next, since a valid `submission.csv` must always be produced within the 8.5h ceiling.

## Inference & Submission

Load the **best** checkpoint (lowest validation CER, not necessarily the last epoch), run batched greedy CTC decoding on all 1,894 test clips, then build and verify `submission.csv`.

In [10]:
# SECTION 20: LOAD BEST + INFERENCE
print(f"\n[INFERENCE] Loading best model from {CKPT_DIR}...")
del model
torch.cuda.empty_cache(); gc.collect()
print_gpu("after del")

model_inf  = Wav2Vec2ForCTC.from_pretrained(CKPT_DIR).to(DEVICE)
proc_inf   = Wav2Vec2Processor.from_pretrained(CKPT_DIR)
tok_inf    = proc_inf.tokenizer
model_inf.eval()
print_gpu("inference ready")
print("  ✓ Best model loaded")

# SECTION 21: TEST INFERENCE
test_preds=[]; success=0; failed=0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Inference"):
        input_values   = batch["input_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        try:
            with torch.cuda.amp.autocast():
                logits = model_inf(
                    input_values=input_values,
                    attention_mask=attention_mask,
                ).logits
            pred_ids = torch.argmax(logits, dim=-1)
            preds    = proc_inf.batch_decode(pred_ids)
            test_preds.extend(preds); success+=len(preds)
        except Exception as ex:
            print(f"  [WARN] {ex}")
            test_preds.extend([""]*input_values.shape[0])
            failed+=input_values.shape[0]

print(f"\n  Success:{success} | Failed:{failed}")
test_preds = (test_preds+[""]*len(test_df))[:len(test_df)]

# SECTION 22: SUBMISSION
submission = pd.DataFrame({
    "ID":            test_df["ID"].values,
    "transcription": test_preds,
})
SUB_PATH = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(SUB_PATH, index=False)

# SECTION 23: VERIFY
chk = pd.read_csv(SUB_PATH)
print("\n"+"="*60)
print("  SUBMISSION VERIFICATION")
print("="*60)
print(f"  Path    : {SUB_PATH}")
print(f"  Shape   : {chk.shape}")
print(f"  Columns : {list(chk.columns)}")
print(f"  Rows    : {len(chk)} (expected {len(test_df)})")
print(f"  NaN     : {chk['transcription'].isna().sum()}")
print(f"  ID match: {(chk['ID'].values==test_df['ID'].values).all()}")
print(f"\n  First 10 rows:")
print(chk.head(10).to_string(index=False))
assert list(chk.columns)==["ID","transcription"]
assert len(chk)==len(test_df)
assert chk["transcription"].isna().sum()==0
assert (chk["ID"].values==test_df["ID"].values).all()
print(f"\n  ✓ All checks passed!")
print(f"  ✓ Best Val CER = {best_cer:.4f}")
print(f"  ✓ Total notebook time = {elapsed_hours():.2f}h "
      f"(hard cap: {HARD_STOP_HOURS}h)")  # --- TIME BUDGET ADDITION ---



[INFERENCE] Loading best model from /kaggle/working/best_mms...
  [GPU after del] alloc=3.76GB  reserved=4.12GB  free≈11.52GB  total=15.64GB


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

  [GPU inference ready] alloc=5.02GB  reserved=5.09GB  free≈10.55GB  total=15.64GB
  ✓ Best model loaded


Test Inference:   0%|          | 0/474 [00:00<?, ?it/s]


  Success:1894 | Failed:0

  SUBMISSION VERIFICATION
  Path    : /kaggle/working/submission.csv
  Shape   : (1894, 2)
  Columns : ['ID', 'transcription']
  Rows    : 1894 (expected 1894)
  NaN     : 0
  ID match: True

  First 10 rows:
                              ID                                                                                                 transcription
f068a206b84c4632865e0629a1b62fb8                                                    bu dorini helila qaynatqan caqqan bol vissiqida viciwalGin
a9d8cfab47b34f12b8f4b4769075713e                                          yamGurdin keyinki hawa Huddi sUzUp tazilanGandAk vintayin sap bolidu
34147b4f995144288b720d7474ba4dd6                                            qar barGancA qattiq yaGdi yolidiki piyadilAr teHimu xalaNxip kAtti
c6c201bcd81a402385c2f008983f7474 cAtkA ciqip bilim vigAnligAndin keyin qaytip kelimAn tAxkilniN maNa bolGan vixAncisini hArgiz yArdA qoymaymAn
c3c190cc67c14d4a946ef1b722196248                